In [ ]:
import json
import pandas as pd
import folium
from itertools import combinations
from collections import defaultdict
from datashader.bundling import hammer_bundle

# ==========================================
# 1. PRÉPARATION DES DONNÉES
# ==========================================
with open('../data/data.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
df = pd.DataFrame(data)

col_etab = 'etablissement_norm' if 'etablissement_norm' in df.columns else 'etablissement'
df = df.dropna(subset=['lat', 'lon', col_etab])

# ==========================================
# 2. CRÉATION DES NOEUDS (Établissements)
# ==========================================
# Datashader a besoin de nœuds avec les colonnes : id, x, y
etab_coords = df.groupby(col_etab)[['lat', 'lon']].mean().reset_index()
nodes = etab_coords.rename(columns={col_etab: 'name', 'lon': 'x', 'lat': 'y'})
nodes['id'] = range(len(nodes))

# Dictionnaire pour retrouver l'ID à partir du nom
node_id_map = nodes.set_index('name')['id'].to_dict()

# Compter les thèses pour la taille des points
etab_sizes = df[col_etab].value_counts().to_dict()

# ==========================================
# 3. CRÉATION DES LIENS (Directeurs communs)
# ==========================================
df_exp = df.explode('directeurs').dropna(subset=['directeurs'])
dir_etabs = df_exp.groupby('directeurs')[col_etab].unique()

edges_weight = defaultdict(int)

for directeur, etablissements in dir_etabs.items():
    if len(etablissements) > 1:
        for pair in combinations(sorted(etablissements), 2):
            edges_weight[pair] += 1

# Datashader a besoin de liens avec les colonnes : source, target, weight
edges_list = []
for (e1, e2), weight in edges_weight.items():
    if e1 in node_id_map and e2 in node_id_map:
        edges_list.append({
            'source': node_id_map[e1],
            'target': node_id_map[e2],
            'weight': weight
        })

edges = pd.DataFrame(edges_list)

# ==========================================
# 4. ALGORITHME DE EDGE BUNDLING (2D)
# ==========================================
print("Calcul des trajectoires de Edge Bundling en cours... (cela peut prendre quelques secondes)")
# hammer_bundle génère un DataFrame avec les coordonnées x, y des lignes fusionnées
bundled = hammer_bundle(nodes, edges)

# Datashader sépare chaque ligne par une ligne de 'NaN'. 
# On va découper ce DataFrame pour pouvoir les tracer dans Folium.
paths = []
current_path = []

for _, row in bundled.iterrows():
    if pd.isna(row['x']):
        if current_path:
            paths.append(current_path)
            current_path = []
    else:
        # Folium utilise le format [latitude, longitude] donc [y, x]
        current_path.append([row['y'], row['x']])

if current_path:
    paths.append(current_path)

# ==========================================
# 5. GÉNÉRATION DE LA CARTE INTERACTIVE
# ==========================================
carte = folium.Map(location=[46.603354, 1.888334], zoom_start=6, tiles='CartoDB dark_matter')

# A. Tracer les lignes fusionnées (Edge Bundling)
for p in paths:
    folium.PolyLine(
        locations=p,
        color='#00e5ff', # Un beau cyan fluo
        weight=1.5,
        opacity=0.3, # L'opacité faible permet de créer un effet lumineux quand les lignes se superposent
        smooth_factor=1
    ).add_to(carte)

# B. Placer les établissements (Nœuds)
for idx, row in nodes.iterrows():
    nom = row['name']
    size = etab_sizes.get(nom, 1)
    rayon = min(max(size * 0.4, 3), 15)
    
    folium.CircleMarker(
        location=[row['y'], row['x']],
        radius=rayon,
        color='#ffffff',
        weight=1,
        fill=True,
        fill_color='#e74c3c',
        fill_opacity=0.9,
        tooltip=f"<b>{nom}</b><br>{size} thèses encadrées"
    ).add_to(carte)

# ==========================================
# 6. SAUVEGARDE
# ==========================================
fichier_sortie = 'carte_edge_bundling.html'
carte.save(fichier_sortie)
print(f"✅ Carte avec Edge Bundling générée avec succès : {fichier_sortie}")

Calcul des trajectoires de Edge Bundling en cours... (cela peut prendre quelques secondes)


ValueError: Custom tiles must have an attribution.